# <span style="color:blue;font-size:1.1em;font-weight:bold">Hypothesis Testing Introduction</span>

Let's do a few examples to see how to perform a hypothesis test in Python and the datahub. 

## Initialization Code Block

<span style="color:red;font-size:1.1em;font-weight:bold">Do not change the cell below, only execute it.</span>

In [2]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import seaborn as sns
import scipy.stats as stats
from scipy.stats import gaussian_kde

## Read in some CSV data files from the web.
sky = Table.read_table('skyscrapers_v2.csv')
top_movies = Table.read_table('top_movies_2017.csv')
pers = Table.read_table('http://faculty.ung.edu/rsinn/personality.csv')

## Helper Functions: Box, Describe, and Density

<span style="color:red;font-size:1.1em;font-weight:bold">Run the cell below to activate our helper functions.</span>`m

In [3]:
def describe(data, var = None):
  if isinstance(data, Table):
    data = data.column(var)
  names = np.array(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  values = np.array([round(np.mean(data),1), round(stats.tstd(data),2), len(data) - 1, "", min(data), np.percentile(data, 25), np.median(data), np.percentile(data, 75), max(data), stats.iqr(data)])
  tab = Table().with_columns('Names', names, 'Values', values)  ;  transposed_tab = Table().with_columns( zip(tab.column('Names'), tab.column('Values')) ).show()
  return

def describes(data, num, group):
  dat = data.column(num)  ;  group_var = data.column(group)  ;  groups = np.unique(group_var)  ;  rows = []  ;  tab = Table(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  for g in groups:
    # Filter data for the current group
    sub_dat = dat[group_var == g]  ;    rows = [g, round(np.mean(sub_dat), 1), round(stats.tstd(sub_dat), 2), len(sub_dat) - 1, min(sub_dat), np.percentile(sub_dat, 25), np.median(sub_dat), np.percentile(sub_dat, 75), max(sub_dat), stats.iqr(sub_dat)]        
    tab = tab.with_row(rows)
  tab.show()
  return

def box(data, num = None, sm = False):
  if isinstance(data, Table):
    df = data.to_df()  ;  sns.boxplot(data=df, y=num, width=0.45, showmeans=sm)  ;
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()  
  else:
    if num == True:
      sm = True  ;  num = 'Variable'
    if num == None:
      num = 'Variable'
    plots.figure(figsize=(6, 6))  ;  plots.boxplot(data, widths = 0.45, showmeans=sm)
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()
  return

def boxes(data, num, cat, sm = False):
  if not isinstance(data, Table):
    print("ERROR: First input must be table name.")
  else:
    df = data.to_df()  ;  sns.boxplot(data=data, x=cat, y=num, width=0.75, showmeans=sm)
    plots.title(f'Boxplot of {num} Grouped by {cat}')  ;  plots.show()
  return

def density(data, num = None):
  if isinstance(data, Table):
    sns.kdeplot( data = data.column(num), bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  else:
    if num == None:
      num = 'Variable'  ;  sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
    else:
      sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  return

def densities(tab, var, group_var):
  groups = tab.column(group_var) ;  unique_groups = np.unique(groups)  ;  plots.figure()
  for g in unique_groups:
    # Filter data for the current group
    mask = (groups == g)
    x = tab.column(var)[mask]   
    # Calculate KDE for the group subset
    kde = gaussian_kde(x)
    x_grid = np.linspace(x.min() - 1, x.max() + 1, 512)  ;    plots.plot(x_grid, kde(x_grid), label=str(g))   
  plots.title(f'Density Plot of {var} by {group_var}')  ;  plots.xlabel(f'{var}')  ;  plots.ylabel('Density')
  plots.legend()  ;  plots.show()
  return

## <span style="color:blue;font-size:1.1em;font-weight:bold">Hypothesis Testing</span> 

From the **Lady Tasting Tea** story, we can pull out these valuable insights:

1. **Null Hypothesis:** A probability model to compare to real-world data.
2. **Cutoff Value:** A low probability cutoff at which point we decide the null is not likely to be true (e.g. to fit the real-world data).
3. **Scientific Conclusions:** We have two statistical conclusions we can make:
  - Reject the null.
  - Fail to reject the null.

This allows us a way to perform calculations and evaluate real-world scientific data to either confirm or debunk research hypotheses.

## <span style="color:blue;font-size:1.1em;font-weight:bold">Example 1: Coping Humor</span> 

We know based upon prior research that the average Coping Humor Score (CHS) at UNG is 25. Test whether biological males have higher levels of Coping Humor than females do (personality table **pers**, column title **CHS**).

## <span style="color:blue;font-size:1.1em;font-weight:bold">Example 2: Paramount vs. Warner Brothers</span> 

Do the movie revenues differ at Paramount compared to Warner Brothers? Test at the 0.05 level using the **top_movies** table and the column title **Gross (Adjusted)**.

## Practice Problems
1. Test whether biological males or females have higher levels of narcissism at the 0.05 level. (Personality table **pers**, column **Narc**).

2. Test whether those involved in social Greek fraternities and soririties are more extroverted than their non-Greek peers. Test at the 0.05 level (Personality table **pers**, column **Extro**).

3. Test whether skyscrapers in Atlanta are taller than those in Los Angeles. Test at the 0.5 level given that a new table has been created from the *sky* table called **atl_la** containing exactly the information we need.

In [8]:
atl_la = sky.where('city', are.contained_in('Atlanta Los Angeles'))
#atl_la